In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch


In [ ]:
import os
import re
import json
import torch
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

# -------------------------------
# 🔐 AUTH
# -------------------------------
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# -------------------------------
# 🤖 MODEL LOAD
# -------------------------------
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

# -------------------------------
# 🧠 TEXT GENERATION (FAST)
# -------------------------------
def generate_text(prompt, max_tokens=400):
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,   
        temperature=0.4,            
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)[len(text):].strip()

# -------------------------------
# 🔍 JSON EXTRACTION
# -------------------------------
def extract_objects(raw):
    # ✅ 1. Try to grab a full JSON array first
    array_match = re.search(r"\[.*?\]", raw, re.DOTALL)
    if array_match:
        try:
            parsed = json.loads(array_match.group())
            if isinstance(parsed, list):
                return [r for r in parsed if isinstance(r, dict)]
        except json.JSONDecodeError:
            pass

    # ✅ 2. Fallback: greedy match for individual objects (handles nested keys)
    matches = re.findall(r"\{[^{}]*\}", raw, re.DOTALL)  
    data = []
    for m in matches:
        try:
            data.append(json.loads(m))
        except json.JSONDecodeError:
            continue
    return data

# -------------------------------
# ⚡ BATCH GENERATION (FAST)
# -------------------------------
CITIES = ["Mumbai", "Delhi", "Pune", "Bangalore", "Chennai", "Hyderabad", "Kolkata", "Jaipur", "Ahmedabad", "Surat"]
CATEGORIES = ["Electronics", "Fashion", "Grocery", "Books", "Sports", "Beauty", "Home Decor", "Toys", "Appliances"]
PAYMENTS = ["UPI", "Credit Card", "Debit Card", "Net Banking", "Cash on Delivery", "Wallet"]

def generate_batch(batch_size=5):
    # 🔥 Pick random hints to seed variety — don't let the model freewheel
    import random
    city_hint = random.sample(CITIES, min(batch_size, len(CITIES)))
    category_hint = random.sample(CATEGORIES, min(batch_size, len(CATEGORIES)))

    prompt = f"""Return ONLY a JSON array with exactly {batch_size} records. No explanation. No markdown.

Rules:
- Every "name" must be a DIFFERENT real Indian full name. Do NOT repeat names.
- Use these cities (one per record): {city_hint}
- Use these categories (one per record): {category_hint}
- "age" must be between 21 and 55
- "total_orders" must be between 1 and 20
- "avg_order_value" must be between 500.0 and 9000.0

Output format (use DIFFERENT values, this is just the shape):
[{{"name":"<first> <last>","age":<int>,"city":"<city>","total_orders":<int>,"avg_order_value":<float>,"preferred_category":"<category>","payment_method":"<payment>"}}]"""

    raw = generate_text(prompt, max_tokens=500)
    raw = re.sub(r"```(?:json)?|```", "", raw).strip()
    return extract_objects(raw)


def generate_dataset(n=100):
    data = []
    seen_names = set()        # 🔥 NEW: block duplicate names
    max_attempts = n * 6
    attempts = 0

    while len(data) < n and attempts < max_attempts:
        batch = generate_batch(5)
        attempts += 1

        for record in batch:
            if len(data) >= n:
                break

            name = record.get("name", "").strip()

            if not name or name in seen_names:
                continue                          # 🔥 skip Rahul Sharma clones

            record["customer_id"] = f"CUST_{len(data) + 1:04d}"
            data.append(record)
            seen_names.add(name)

        print(f"Generated: {len(data)} / {n}  (attempt {attempts})")

    if len(data) < n:
        print(f"⚠️  Only got {len(data)} unique records — model too repetitive for n={n}")

    return data[:n]

# -------------------------------
# 💾 SAVE DATASET
# -------------------------------
if __name__ == "__main__":
    data = generate_dataset(10)   # 🔥 keep small for speed

    df = pd.DataFrame(data)
    df.to_csv("synthetic_data.csv", index=False)

    print("\n✅ Done! Sample data:")
    print(df.head())